In [ ]:
#Step 0: Import Libraries & Global Settings
# Core data analysis and manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Data preprocessing
from sklearn.preprocessing import StandardScaler
from scipy import stats

# Clustering models
from sklearn.cluster import KMeans, AgglomerativeClustering

# Clustering evaluation metrics
from sklearn.metrics import silhouette_score

# Hierarchical clustering visualization
import scipy.cluster.hierarchy as sch

# Global plot settings (no Chinese garble, style)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False  # Keep minus sign normal
sns.set_style('whitegrid')

Step 1: Load Data & Handle Missing/Outliers (1 point)
1.1 Load Dataset & Basic Inspection

In [ ]:
# Load dataset (replace with xlsx if your file is .xlsx: pd.read_excel())
df = pd.read_csv("national_anthems.csv")

# Basic data inspection
print("=== Basic Data Information ===")
print(df.head())  # First 5 rows
print("\n=== Data Structure & Missing Values ===")
print(df.info())
print("\n=== Descriptive Statistics for Numerical Features ===")
print(df.describe())

# Define numerical features (exclude non-analytical columns like country/id)
num_features = df.select_dtypes(include=[np.float64, np.int64]).columns.tolist()
num_features = [col for col in num_features if col not in ['country', 'id', 'Country', 'ID']]
print(f"\n=== Numerical Features for Clustering ===")
print(num_features)

1.2 Detect & Process Missing Values

In [ ]:
# Calculate missing value rate for each column
missing_rate = (df.isnull().sum() / len(df)) * 100
missing_features = missing_rate[missing_rate > 0]

if not missing_features.empty:
    print("\n=== Missing Value Rate ===")
    print(missing_features)
    # Option 1: Drop rows with missing values (recommended if missing rate <5%)
    df = df.dropna()
    # Option 2: Impute missing values (median for numerical, mode for categorical)
    # df[num_features] = df[num_features].fillna(df[num_features].median())
else:
    print("\n=== No Missing Values in Dataset ===")

# Reset index after dropping rows
df = df.reset_index(drop=True)

1.3 Detect & Process Outliers (IQR Method)

In [ ]:
# Outlier detection with IQR (robust to extreme values)
def remove_outliers(df, features):
    for feat in features:
        Q1 = df[feat].quantile(0.25)
        Q3 = df[feat].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        # Filter out outliers
        df = df[(df[feat] >= lower_bound) & (df[feat] <= upper_bound)]
    return df

# Remove outliers
df_clean = remove_outliers(df, num_features)
print(f"\n=== Data Volume After Outlier Removal ===")
print(f"Original: {len(df)} rows | Cleaned: {len(df_clean)} rows")

# Visualize outliers with boxplots (for report)
plt.figure(figsize=(12, 8))
for i, feat in enumerate(num_features):
    plt.subplot(2, len(num_features)//2 + 1, i+1)
    sns.boxplot(x=df_clean[feat], color='lightblue')
    plt.title(f'Boxplot - {feat} (No Outliers)')
plt.tight_layout()
plt.savefig('01_outlier_boxplot.png')  # Save plot for report
plt.show()

# Visualize numerical feature distribution (histogram + KDE)
plt.figure(figsize=(12, 8))
for i, feat in enumerate(num_features):
    plt.subplot(2, len(num_features)//2 + 1, i+1)
    sns.histplot(df_clean[feat], kde=True, color='skyblue')
    plt.title(f'Distribution - {feat}')
plt.tight_layout()
plt.savefig('02_numerical_feature_distribution.png')
plt.show()

Step 2: Exploratory Data Analysis (EDA) (3 points)
2.1 Univariate & Bivariate Analysis

In [ ]:
# 1. Correlation heatmap for numerical features
corr_matrix = df_clean[num_features].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f')
plt.title('Correlation Heatmap of Numerical Features')
plt.tight_layout()
plt.savefig('03_correlation_heatmap.png')
plt.show()

# 2. Categorical feature analysis (if any, e.g., region, language)
cat_features = df_clean.select_dtypes(include=[object, 'category']).columns.tolist()
cat_features = [col for col in cat_features if col not in ['country', 'id', 'Country', 'ID']]

if cat_features:
    for feat in cat_features:
        # Category proportion pie chart
        plt.figure(figsize=(8, 6))
        df_clean[feat].value_counts().plot(kind='pie', autopct='%1.1f%%', startangle=90, colors=sns.color_palette('pastel'))
        plt.title(f'Category Proportion - {feat}')
        plt.ylabel('')
        plt.savefig(f'04_categorical_proportion_{feat}.png')
        plt.show()

        # Category vs numerical feature (boxplot)
        for num_feat in num_features[:2]:  # Plot top 2 numerical features for brevity
            plt.figure(figsize=(10, 6))
            sns.boxplot(x=df_clean[feat], y=df_clean[num_feat], palette='pastel')
            plt.title(f'{num_feat} by {feat}')
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.savefig(f'05_{num_feat}_by_{feat}.png')
            plt.show()
else:
    print("\n=== No Categorical Features in Dataset ===")

# EDA Summary (copy this to your report and modify with your data insights)
print("""
=== EDA Summary Template ===
1. Distribution: [Feature1] is right-skewed with mean X and median Y; [Feature2] is normally distributed.
2. Correlation: [FeatureA] and [FeatureB] have a strong positive correlation (r=0.8); [FeatureC] has no obvious correlation with other features.
3. Categorical: [CategoryFeat] is dominated by [X] (XX%) and [Y] (XX%); [NumericalFeat] is significantly higher in [X] category than [Y].
""")

Step 3: Feature Scaling with StandardScaler (1 point)

In [ ]:
# Extract feature matrix for clustering
X = df_clean[num_features].values

# Standard Scaling (mean=0, std=1) - critical for distance-based clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert scaled data to DataFrame for inspection
X_scaled_df = pd.DataFrame(X_scaled, columns=num_features)
print("=== Scaled Feature Statistics (Mean≈0, Std≈1) ===")
print(X_scaled_df.describe().round(2))

Step 4: Build KMeans & HAC Models (2 points)

In [ ]:
# Set random state for reproducibility
RANDOM_STATE = 42

# 4.1 KMeans Clustering
kmeans_tentative = KMeans(n_clusters=3, init='k-means++', random_state=RANDOM_STATE)
kmeans_labels_tentative = kmeans_tentative.fit_predict(X_scaled)
# Add cluster labels to cleaned dataframe
df_clean['kmeans_cluster_tentative'] = kmeans_labels_tentative

# 4.2 Hierarchical Agglomerative Clustering (HAC) - Single Linkage (per courseware)
hac_tentative = AgglomerativeClustering(n_clusters=3, linkage='single')
hac_labels_tentative = hac_tentative.fit_predict(X_scaled)
df_clean['hac_cluster_tentative'] = hac_labels_tentative

# Inspect tentative cluster distribution
print("\n=== Tentative Cluster Distribution (k=3) ===")
print("KMeans Clusters:", df_clean['kmeans_cluster_tentative'].value_counts().sort_index())
print("HAC Clusters (Single Linkage):", df_clean['hac_cluster_tentative'].value_counts().sort_index())

# Save tentative clustered data (for report)
df_clean.to_csv('06_tentative_clustered_data.csv', index=False)

Step 5: Find Optimal Number of Clusters (k) (1 point)

5.1 Elbow Method (Core - per courseware)

In [ ]:
# Calculate inertia (within-cluster sum of squares) for k=1 to 10
inertia_list = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=RANDOM_STATE)
    kmeans.fit(X_scaled)
    inertia_list.append(kmeans.inertia_)

# Plot Elbow Curve
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia_list, marker='o', linestyle='-', color='crimson', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (Within-Cluster Sum of Squares)')
plt.title('Elbow Method - Optimal k Selection')
plt.xticks(k_range)
plt.grid(True, alpha=0.3)
plt.savefig('07_elbow_method_curve.png')
plt.show()

# 5.2 Silhouette Score (Supplementary - for better k validation)
sil_score_list = []
k_range_sil = range(2, 11)  # Silhouette score is invalid for k=1

for k in k_range_sil:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=RANDOM_STATE)
    labels = kmeans.fit_predict(X_scaled)
    sil_score = silhouette_score(X_scaled, labels)
    sil_score_list.append(sil_score)

# Plot Silhouette Score Curve
plt.figure(figsize=(10, 6))
plt.plot(k_range_sil, sil_score_list, marker='s', linestyle='-', color='forestgreen', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score - Optimal k Selection')
plt.xticks(k_range_sil)
plt.grid(True, alpha=0.3)
plt.savefig('08_silhouette_score_curve.png')
plt.show()

# Get optimal k (MANUALLY SET THIS BASED ON YOUR PLOTS, e.g., optimal_k=2)
optimal_k = 2  # <<< MODIFY THIS VALUE ACCORDING TO YOUR ELBOW/SILHOUETTE PLOTS
print(f"\n=== Optimal Number of Clusters Selected ===")
print(f"Optimal k = {optimal_k} (based on Elbow Method + Silhouette Score)")

# Re-train KMeans and HAC with OPTIMAL k
# Optimal KMeans
kmeans_opt = KMeans(n_clusters=optimal_k, init='k-means++', random_state=RANDOM_STATE)
df_clean['kmeans_cluster_opt'] = kmeans_opt.fit_predict(X_scaled)

# Optimal HAC (Single Linkage)
hac_opt = AgglomerativeClustering(n_clusters=optimal_k, linkage='single')
df_clean['hac_cluster_opt'] = hac_opt.fit_predict(X_scaled)

# Inspect optimal cluster distribution
print("\n=== Optimal Cluster Distribution (k={}) ===".format(optimal_k))
print("KMeans Optimal Clusters:", df_clean['kmeans_cluster_opt'].value_counts().sort_index())
print("HAC Optimal Clusters (Single Linkage):", df_clean['hac_cluster_opt'].value_counts().sort_index())

# Save optimal clustered data
df_clean.to_csv('09_optimal_clustered_data.csv', index=False)

Step 6: Final Visualization & Analysis (2 points + Extra Credit 2 points)


6.1 HAC Dendrogram (Extra Credit - per courseware)

In [ ]:
# Plot HAC Dendrogram (Single Linkage)
plt.figure(figsize=(12, 8))
sch.dendrogram(sch.linkage(X_scaled, method='single'))
# Add a horizontal line for optimal k (MODIFY THE Y VALUE BASED ON YOUR DENDROGRAM)
plt.axhline(y=2.0, color='red', linestyle='--', label=f'Optimal k={optimal_k}')
plt.xlabel('Samples (Countries)')
plt.ylabel('Distance')
plt.title('Hierarchical Agglomerative Clustering - Dendrogram (Single Linkage)')
plt.legend()
plt.tight_layout()
plt.savefig('10_hac_dendrogram.png')
plt.show()

6.2 Cluster Feature Analysis (for Step6 Discussion)

In [ ]:
# Analyze numerical feature mean by optimal KMeans clusters
kmeans_cluster_analysis = df_clean.groupby('kmeans_cluster_opt')[num_features].mean().round(2)
print("\n=== KMeans Optimal Cluster - Feature Mean ===")
print(kmeans_cluster_analysis)

# Analyze numerical feature mean by optimal HAC clusters
hac_cluster_analysis = df_clean.groupby('hac_cluster_opt')[num_features].mean().round(2)
print("\n=== HAC Optimal Cluster - Feature Mean ===")
print(hac_cluster_analysis)

# Visualize cluster distribution (2D - use top 2 correlated features)
# Select 2 features for scatter plot (modify based on your correlation heatmap)
feat1, feat2 = num_features[0], num_features[1]

# KMeans cluster scatter plot
plt.figure(figsize=(10, 6))
sns.scatterplot(x=df_clean[feat1], y=df_clean[feat2], hue=df_clean['kmeans_cluster_opt'], 
                palette='viridis', s=80, edgecolor='black')
plt.title(f'KMeans Clustering (k={optimal_k}) - {feat1} vs {feat2}')
plt.xlabel(feat1)
plt.ylabel(feat2)
plt.legend(title='Cluster')
plt.savefig(f'11_kmeans_cluster_{feat1}_vs_{feat2}.png')
plt.show()

# HAC cluster scatter plot
plt.figure(figsize=(10, 6))
sns.scatterplot(x=df_clean[feat1], y=df_clean[feat2], hue=df_clean['hac_cluster_opt'], 
                palette='plasma', s=80, edgecolor='black')
plt.title(f'HAC Clustering (k={optimal_k}, Single Linkage) - {feat1} vs {feat2}')
plt.xlabel(feat1)
plt.ylabel(feat2)
plt.legend(title='Cluster')
plt.savefig(f'12_hac_cluster_{feat1}_vs_{feat2}.png')
plt.show()

# Step6 Discussion Template (copy to report and modify)
print(f"""
=== Step6 Discussion Template ===
1. Data Preprocessing: Missing values ({missing_features.sum() if 'missing_features' in locals() else 0}) were removed; outliers filtered by IQR retained {len(df_clean)/len(df)*100:.1f}% of data.
2. EDA: [Key correlation/distribution insights from Step2].
3. Feature Scaling: StandardScaler normalized all features to mean=0, std=1, eliminating unit bias for clustering.
4. Model Training: Tentative k=3 showed unbalanced clusters; optimal k={optimal_k} was selected via Elbow/Silhouette.
5. Cluster Analysis: KMeans Cluster 0 has the highest [FeatureX] mean; HAC Cluster 1 is dominated by low [FeatureY] values.
6. Model Comparison: KMeans produces balanced global clusters; HAC (Single Linkage) forms dense local clusters (chaining effect).
7. Practical Insight: National anthem [FeatureA] is highly correlated with [country characteristic], clustering aligns with geographic/language patterns.
""")